## 1. Importar librerías y archivos de datos

### Instalar e importar librerías:

**Importar librerías** necesarias:

In [21]:
# Importar librerías de Python necesarias
import pandas as pd
import re

**Importar y concatenar archivos de datos** (en CSV o Excel):

In [22]:
# Load data
df_ice = pd.read_csv('../data/raw/ice.csv')
df_lgbt = pd.read_csv('../data/raw/lgbt.csv')
df_islamophobia = pd.read_csv('../data/raw/islamophobia.csv')
df_asianhate = pd.read_csv('../data/raw/asianhate.csv')
df_pal_isr = pd.read_csv('../data/raw/pal_isr.csv')

### Previsualizar datos importados:

**Previsualizar tabla** con todos los registros importados:

In [23]:
# Previsualizar dataframe con CSVs importados
display(df_asianhate)
print(f"Filas/Columnas (shape) en registros importados: {df_asianhate.shape}")

,author,body,datalegacylang
0,DorcasOrcas,"China, this is what you are known for:\n#anima...",en
1,RodneyR58127664,#BoycottChina #CatAbusersChina visa workers an...,en
2,nimernight38961,#China\n#MadeInChina\n#BoycottChina\n#ChinaTra...,en
3,fish_puddle,@Samesamaalways @nimernight38961 They are ugly...,en
4,RamirezVelhagen,#BoycottChina and its culture of cruelty and b...,en
...,...,...,...
627,ChiangKaiShek81,ISIS parroting Beijing’s narrative? Convenient...,en
628,wangleehom03,"Beijing’s trade tactics: hack the critics, hij...",en
629,wangleehom03,"While China destroys Tibetan identity, leaders...",en
630,liwan003,From cancer cures to defense tech—China’s reac...,en


Filas/Columnas (shape) en registros importados: (632, 3)


**Exportar copia en CSV con registros importados (o concatenados):**

---

## 2. Limpieza de registros importados

In [24]:
hashtags_ban = [
    "DeportThemAll", "MassDeportationsNow", "ICE", "GenderIdeology",
    "ParentsRights", "WokeMindVirus", "NoLGBT", "BanIslam",
    "IslamIsTheProblem", "StopIslam", "StopIslamInvasion",
    "BoycottChina", "ChinaVirus", "FuckChina", "StopChina",
    "FreePalestine", "HamasTerrorists", "StandWithIsrael"
]


hashtags_ban_lower = [h.lower() for h in hashtags_ban]



def limpiar_texto(texto):
    if pd.isna(texto):
        return texto


    texto = re.sub(r'\r\n|\r|\n', ' ', texto)


    texto = texto.replace('\\n', ' ')

    texto = re.sub(r'http\S+|www\S+', '', texto)


    texto = re.sub(r'@\w+', '', texto)


    def procesar_hashtag(match):
        palabra = match.group(1)
        if palabra.lower() in hashtags_ban_lower:
            return ''
        return palabra
    texto = re.sub(r'#(\w+)', procesar_hashtag, texto)


    texto = re.sub(r'[^\x00-\x7F]+', '', texto)


    texto = texto.lower()


    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto



def limpiar_y_deduplicar(df, columna):

    df_sin_dup_antes = df.drop_duplicates(subset=[columna], keep='first').copy()

    df_sin_dup_antes['clean_text'] = df_sin_dup_antes[columna].apply(limpiar_texto)


    df_final = df_sin_dup_antes.drop_duplicates(subset=['clean_text'], keep='first').reset_index(drop=True)


    df_eliminados = df_sin_dup_antes[df_sin_dup_antes.duplicated(subset=['clean_text'], keep='first')]

    return df_final, df_eliminados

In [25]:
df_final1, df_eliminados1 = limpiar_y_deduplicar(df_ice, 'body')
df_final2, df_eliminados2 = limpiar_y_deduplicar(df_lgbt, 'body')
df_final3, df_eliminados3 = limpiar_y_deduplicar(df_asianhate, 'body')
df_final4, df_eliminados4 = limpiar_y_deduplicar(df_islamophobia, 'body')
df_final5, df_eliminados5 = limpiar_y_deduplicar(df_pal_isr, 'body')

In [26]:
df_final1

,author,body,datalegacylang,clean_text
0,Victory202424,@01CuriousGeorge Shaking my head…..,en,shaking my head..
1,FredKrueger27,@01CuriousGeorge If he's providing proper inst...,en,"if he's providing proper instruction, i could ..."
2,TimeHackerVicki,@01CuriousGeorge What's he doing wrong? Speaki...,en,what's he doing wrong? speaking hindi?
3,YoungBobTPUK,Radical Muslim recongnizes me and throughout t...,en,radical muslim recongnizes me and throughout t...
4,Acts17David,@YoungBobTPUK Were this guy's parents first co...,en,were this guy's parents first cousins who also...
...,...,...,...,...
2774,michaelasquith,@davidaxelrod @jimmykimmel @FLOTUS45\‚ please....,en,\ please.. then maybe your husband should stop...
2775,BluebonnetsTx8,@DAGToddBlanche But you don’t stand for peop...,en,but you dont stand for people who have the rig...
2776,IBTimesUK,Fear becomes a weapon as scammers exploit frag...,en,fear becomes a weapon as scammers exploit frag...
2777,WhatSilence,@TheJFreakinC why is my country being used to ...,en,why is my country being used to incite hate/vi...


In [28]:
df_final1.to_csv('../data/interim/ice_clean.csv', index=False)
df_final2.to_csv('../data/interim/lgbt_clean.csv', index=False)
df_final3.to_csv('../data/interim/asianhate_clean.csv', index=False)
df_final4.to_csv('../data/interim/islamophobia_clean.csv', index=False)
df_final5.to_csv('../data/interim/pal_isr_clean.csv', index=False)